# Mixture of Experts (MoE) - 실습 코드 1: MoE Layer 구현 (PyTorch)

- Tutorial ID: `expand-moe`
- Tutorial: Mixture of Experts (MoE)
- Section ID: `expand-moe-code-1`
- Section: 실습 코드 1: MoE Layer 구현 (PyTorch)

이 노트북은 Mixture of Experts(MoE) 레이어를 PyTorch로 처음부터 직접 구현하면서,
Transformer의 FFN(Feed-Forward Network) 자리에 여러 개의 작은 전문가(Expert)
네트워크를 어떻게 끼워 넣는지 한 단계씩 따라가 봅니다.

**이런 분들을 위한 노트북입니다**
- MoE라는 단어는 들어봤지만 텐서가 실제로 어떻게 움직이는지는 처음 보는 분
- Router, Top-k, Expert, Shared Expert 같은 용어를 코드와 함께 눈으로 확인하고 싶은 분
- Transformer의 attention/FFN 정도는 알고 있지만 MoE는 처음 다뤄보는 분

**미리 준비할 것**: PyTorch가 설치된 환경 (Colab을 쓰면 별도 설치 없이 바로 실행됩니다)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: MoE Layer 구현 (PyTorch)
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# MoE(Mixture of Experts)라는 개념이 실제 텐서 연산으로
# 어떻게 바뀌는지 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 토큰 x가 라우터(Router)를 거쳐 "전문가별 점수(logit)"로 바뀌는 과정 추적
#   2) 전체 전문가 중 top-k개만 고르고, 고른 것들만 softmax로 정규화해
#      "가중치"를 만드는 과정 관찰 (선택되지 않은 전문가는 가중치 0)
#   3) 각 전문가가 "자기 담당 토큰만" 골라서 계산하는 sparse 연산 흐름 확인
#      (전문가가 8명 있어도 top_k=2라면, 토큰 1개당 실제로는 2명만 일합니다)
#   4) 여러 전문가의 출력이 라우팅 가중치로 다시 가중합되는 과정 확인
#   5) Shared Expert(항상 켜져 있는 공용 전문가, DeepSeek-MoE 스타일)가
#      왜/어떻게 더해지는지 이해
#
# 읽는 순서:
#   1) 이 노트북 전체에서 재사용할 작은 예시(토큰 6개, 전문가 4명)를 먼저 확인합니다.
#   2) Router -> Top-k -> Softmax -> Expert -> 합치기 -> Shared Expert 순서로,
#      작은 예시를 손으로 따라가며 각 단계의 shape과 실제 값을 눈으로 봅니다.
#   3) 위 단계들을 하나의 nn.Module(MoELayer)로 합친 "완성된 코드"를 확인합니다.
#   4) 작은 예시와, 실제 크기(d_model=512 등)에 가까운 예시 모두에 완성된
#      코드를 돌려서 shape이 잘 유지되는지 확인합니다.
#   5) num_experts, top_k 등을 바꿔가며 결과(특히 shape과 전문가별 토큰 수)가
#      어떻게 달라지는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - torch에 의존하는 코드이므로 Colab/로컬/서버 등 Python + PyTorch가
#     설치된 환경에서 실행하는 것을 권장합니다.
#   - 이 노트북은 MoE "레이어 구현"에 집중합니다. 실전에서 중요한
#     load balancing loss(전문가 부하를 고르게 만드는 보조 손실 함수),
#     capacity factor(전문가 1명이 처리할 수 있는 토큰 수 제한) 같은 주제는
#     다음 실습에서 이어집니다.
# ============================================================


## 0. 시작하기 전에 — Mixture of Experts(MoE)란?

Transformer 블록 안에는 attention 다음에 **FFN(Feed-Forward Network)** 이라는,
토큰 하나하나를 변환해주는 작은 신경망이 들어갑니다. 보통은 이런 FFN이 레이어마다
**딱 하나**씩 있고, 모든 토큰이 그 하나의 FFN을 통과합니다.

**MoE의 아이디어는 간단합니다.**

> "FFN을 하나만 두지 말고 여러 개(예: 8개) 두자. 대신 토큰 하나당 그 여러 개를
> 다 쓰지 말고, 그중 가장 적합해 보이는 몇 개(예: 2개)만 골라서 쓰자."

**비유하자면 이렇습니다.**
- 기존 FFN 1개 = 내과·외과·피부과를 모두 혼자 보는 "만능 의사" 1명
  → 모든 환자(토큰)를 이 의사 한 명이 진료합니다.
- MoE = 각 분야 전문의가 8명 있는 병원
  → 환자(토큰)가 오면 접수처(Router)가 "이 환자는 A 전문의, B 전문의를 만나면
    되겠다"고 판단해서 8명 중 2명에게만 보냅니다.

이렇게 하면 좋은 점은,

- **전체 파라미터(전문가 수)는 늘어나서 모델이 담을 수 있는 지식/표현력은 커지지만**
- **토큰 1개를 처리할 때 실제로 계산에 참여하는 파라미터(top_k개 전문가)는 크게 늘지 않는다**

는 것입니다. 이 노트북 마지막에서 실제 숫자로 이 부분을 확인해볼 것입니다.

**MoE 레이어를 이루는 4가지 조각**

| 이름 | 역할 |
|---|---|
| Router (라우터) | 토큰마다 "전문가 몇 명 중 누가 적합한가" 점수를 매기는 작은 선형 레이어 |
| Top-k 선택 | 점수가 가장 높은 k명의 전문가만 고르기 |
| Expert (전문가) | 각각 독립적인 작은 FFN. 자기에게 배정된 토큰만 처리 |
| Shared Expert | (선택 사항, DeepSeek-MoE 스타일) 라우팅과 무관하게 **모든** 토큰이 항상 거치는 공용 FFN |

아래에서 이 4가지 조각을 하나씩, 작은 예시로 직접 만들어보겠습니다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 실행할 때마다 난수(랜덤 초기 가중치, 랜덤 입력)가 똑같이 나오도록 시드를 고정합니다.
# -> 이 노트북에 적힌 출력값과 여러분이 직접 실행한 결과가 (환경이 같다면) 동일하게 나옵니다.
torch.manual_seed(42)

# 텐서를 출력할 때 소수점 3자리까지만 보이도록 설정합니다 (가독성을 위한 설정일 뿐, 계산에는 영향 없음)
torch.set_printoptions(precision=3, sci_mode=False)

print(f"PyTorch 버전: {torch.__version__}")


## 이 노트북에서 계속 사용할 작은 예시

앞으로 나오는 모든 단계는 아래와 같은 작은 숫자로 먼저 확인합니다. 숫자가 작아야
`print`로 값을 직접 눈으로 확인할 수 있기 때문입니다.

- 토큰(token) 개수: **6개** — 실제로는 `batch_size * seq_len`만큼의 토큰이 있지만,
  라우팅은 토큰 하나하나에 대해 독립적으로 일어나므로 여기서는 그냥
  "토큰이 6개 있다"고만 생각해도 충분합니다.
- 토큰 하나의 차원 `d_model`: **8**
- 전문가(Expert) 수 `num_experts`: **4**
- 토큰 하나당 선택하는 전문가 수 `top_k`: **2**

노트북 뒷부분에서는 실제 논문/모델에 가까운 크기(`d_model=512`, `num_experts=8` 등)로도
똑같은 코드를 돌려볼 것입니다.


In [ ]:
# 이 노트북 전체에서 재사용할 작은 예시용 하이퍼파라미터
num_tokens_demo = 6    # 토큰 개수 (설명 편의상 batch/seq 구분 없이 "토큰 6개"로 취급)
d_model_demo = 8       # 토큰 벡터의 차원
num_experts_demo = 4   # 전문가 수
top_k_demo = 2         # 토큰 1개당 선택할 전문가 수

# 토큰 6개짜리 가짜 입력을 하나 만듭니다. (실제로는 이전 레이어의 출력이 여기로 들어옵니다)
x_demo = torch.randn(num_tokens_demo, d_model_demo)

print("x_demo.shape:", x_demo.shape)  # (6, 8) -> "토큰 6개, 각각 8차원 벡터"
print(x_demo)


## 1. Router (라우터) — 토큰마다 "전문가별 점수" 매기기

Router는 사실 아주 단순합니다. **입력 차원(`d_model`) → 전문가 수(`num_experts`)**
로 가는 `nn.Linear` 한 층이 전부입니다.

- 입력: 토큰 벡터 (차원 `d_model`)
- 출력: 전문가 수만큼의 숫자(logit). 숫자가 클수록 "이 토큰에는 이 전문가가 잘
  맞겠다"는 뜻입니다.

아직 확률은 아닙니다 (전문가별 숫자를 다 더해도 1이 아닙니다). 그냥 "점수"일
뿐이고, 점수를 가중치로 바꾸는 것은 뒤에서 softmax가 담당합니다.


In [ ]:
router_demo = nn.Linear(d_model_demo, num_experts_demo)

router_logits_demo = router_demo(x_demo)  # (num_tokens, num_experts)

print("router_logits_demo.shape:", router_logits_demo.shape)  # (6, 4)
print(router_logits_demo)
# 행(row) = 토큰, 열(column) = 전문가.
# 예를 들어 0번째 행은 "토큰 0이 전문가 0/1/2/3과 각각 얼마나 잘 맞는지"에 대한 점수입니다.


## 2. Top-k 선택 — 점수가 가장 높은 전문가 k명만 고르기

전문가가 4명 있어도, 토큰 하나당 실제로는 `top_k=2`명만 사용합니다. `torch.topk`는
각 행(토큰)마다 값이 가장 큰 k개와, 그 값들이 원래 몇 번째 열(=몇 번 전문가)이었는지를
함께 돌려줍니다.

```
values, indices = torch.topk(입력, k, dim=-1)
```
- `values`: 가장 큰 k개의 값 자체 (여기서는 선택된 전문가들의 점수)
- `indices`: 그 값들이 원래 몇 번째 위치였는지 (여기서는 선택된 전문가의 번호)


In [ ]:
top_k_vals_demo, top_k_idx_demo = torch.topk(router_logits_demo, top_k_demo, dim=-1)

print("top_k_vals_demo.shape:", top_k_vals_demo.shape)  # (6, 2)
print("top_k_idx_demo.shape :", top_k_idx_demo.shape)   # (6, 2)

print("\n선택된 점수(top_k_vals_demo):\n", top_k_vals_demo)
print("\n선택된 전문가 번호(top_k_idx_demo):\n", top_k_idx_demo)

# 토큰별로 어떤 전문가가 뽑혔는지 사람이 읽기 좋게 출력해봅니다.
for token_id in range(num_tokens_demo):
    chosen_experts = top_k_idx_demo[token_id].tolist()
    chosen_scores = [round(v, 3) for v in top_k_vals_demo[token_id].tolist()]
    print(f"토큰 {token_id}: 전문가 {chosen_experts} 선택 (점수 {chosen_scores})")


## 3. 선택된 점수만 softmax → 합이 1인 가중치로 변환

`top_k_vals_demo`는 아직 "점수"일 뿐 가중치(weight)가 아닙니다. 이것을 각
토큰별로 softmax에 넣어서, **선택된 top_k개끼리** 합이 1이 되는 가중치로
바꿔줍니다.

> 왜 전체 전문가(4명)가 아니라 선택된 top_k개(2명)에만 softmax를 적용할까요?
> → 선택되지 않은 전문가는 애초에 계산에 참여하지 않으므로 가중치도 0이라고
> 보는 것이 맞고, 선택된 전문가들끼리는 "둘을 어떤 비율로 섞을지"만 정하면
> 됩니다. 그래서 top_k개만 놓고 다시 정규화(re-normalize)합니다.


In [ ]:
top_k_weights_demo = F.softmax(top_k_vals_demo, dim=-1)  # (6, 2)

print("top_k_weights_demo:\n", top_k_weights_demo)

# 각 토큰별로, 뽑힌 두 전문가에 대한 가중치의 합이 1인지 확인해봅니다.
print("\n토큰별 가중치 합 (모두 1이어야 정상):")
print(top_k_weights_demo.sum(dim=-1))


## 4. Expert (전문가) — 각각은 작은 FFN

전문가 하나하나는 특별한 게 아니라, Transformer에서 흔히 보는 FFN과 똑같은
구조입니다.

```
d_model 차원 --[Linear]--> d_ff 차원 --[활성화 함수]--> --[Linear]--> 다시 d_model 차원
```

- 중간에서 잠깐 더 넓은 차원(`d_ff`, 보통 `d_model`의 2~4배 정도)으로 갔다가 다시
  원래 차원으로 돌아옵니다.
- 활성화 함수로는 `SiLU`(=Swish)를 사용합니다. `ReLU`처럼 비선형성을 더해주는
  함수인데, 최근 LLM들(Llama 계열 등)에서 즐겨 사용합니다.
- `nn.Sequential`은 안에 나열한 레이어들을 순서대로 통과시켜주는 컨테이너입니다.

전문가가 8명이면 이런 작은 FFN이 8개 존재하고, **서로 다른 가중치**를 가지고
독립적으로 학습됩니다.


In [ ]:
d_ff_demo = 16  # 전문가 내부 hidden 차원 (보통 d_model보다 크게 잡습니다)

# 전문가 1명 = Linear -> SiLU -> Linear
single_expert_demo = nn.Sequential(
    nn.Linear(d_model_demo, d_ff_demo),
    nn.SiLU(),
    nn.Linear(d_ff_demo, d_model_demo),
)

# 잠깐 테스트: 토큰 6개를 모두 이 전문가 1명에게 통과시켜 봅니다.
# (실전에서는 이렇게 "전문가 1명이 토큰 전부"를 보는 일은 없고,
#  자기에게 배정된 일부 토큰만 보게 됩니다 - 바로 다음 단계에서 확인합니다)
sample_expert_output = single_expert_demo(x_demo)
print("sample_expert_output.shape:", sample_expert_output.shape)  # (6, 8) - 입력과 같은 shape


## 5. 전문가별로 "자기 담당 토큰만" 모아서 계산하고, 다시 합치기

지금까지 만든 재료를 정리하면:
- `top_k_idx_demo`: 토큰마다 어떤 전문가 2명이 뽑혔는지 (전문가 번호)
- `top_k_weights_demo`: 그 2명을 각각 얼마나 반영할지 (가중치, 합=1)

이제 실제로 **전문가별로** 계산을 해야 합니다. 핵심 아이디어는 이렇습니다.

> 전문가 0에게는 "전문가 0을 선택한 토큰들"만 모아서 보여주고, 그 전문가의
> 출력에 "그 토큰이 전문가 0에게 준 가중치"를 곱해서 원래 토큰 자리에
> 더해줍니다. 이것을 전문가 0, 1, 2, 3 전부에 대해 반복합니다.

즉 전문가 1명 입장에서는 **자기를 선택한 토큰만 보이고, 나머지 토큰은 아예
존재하지 않는 것처럼** 계산합니다. 이게 바로 MoE가 "sparse(희소)"하다고
불리는 이유입니다 — 전문가가 4명이어도 각 전문가는 자기 몫의 토큰만 처리하므로,
전체 계산량은 "전문가 4명 x 토큰 6개"가 아니라 "top_k(2) x 토큰 6개"에
가깝습니다.

아래 코드에서 전문가 0번부터 3번까지 차례로:
1. 이 전문가를 선택한 토큰이 누구인지 찾고 (`token_mask`)
2. 그 토큰들만 모아서 전문가에 통과시키고 (`expert_output`)
3. 각 토큰이 이 전문가에게 준 가중치를 찾아서 (`combine_weight`)
4. `가중치 x 전문가출력`을 원래 토큰 자리에 더해줍니다 (`output[token_mask] += ...`)


In [ ]:
# 4명의 전문가를 미리 만들어둡니다 (전문가마다 서로 다른 랜덤 가중치를 가짐)
experts_demo = nn.ModuleList([
    nn.Sequential(
        nn.Linear(d_model_demo, d_ff_demo),
        nn.SiLU(),
        nn.Linear(d_ff_demo, d_model_demo),
    )
    for _ in range(num_experts_demo)
])

# 최종 출력을 담을 그릇을 0으로 채워서 준비합니다. (입력과 같은 shape)
output_demo = torch.zeros_like(x_demo)

for expert_id in range(num_experts_demo):
    # (a) 이 전문가가 top-k 안에 든 토큰이 어디인지 찾기
    #     top_k_idx_demo == expert_id  ->  (6, 2) 짜리 True/False
    #     .any(dim=-1)                 ->  토큰마다 "2개 중 하나라도 이 전문가면 True"
    matches = (top_k_idx_demo == expert_id)   # (6, 2)
    token_mask = matches.any(dim=-1)           # (6,)

    num_selected = token_mask.sum().item()
    selected_token_ids = torch.where(token_mask)[0].tolist()
    print(f"[전문가 {expert_id}] 선택한 토큰 수: {num_selected}, 토큰 번호: {selected_token_ids}")

    if num_selected == 0:
        # 이 전문가를 아무도 선택하지 않았다면 계산할 필요가 없으니 건너뜁니다.
        continue

    # (b) 이 전문가를 선택한 토큰들만 모아서 전문가에 통과시킵니다.
    #     -> 나머지 토큰은 이 전문가의 forward에 아예 들어가지 않습니다 (sparse!)
    expert_input = x_demo[token_mask]                       # (num_selected, d_model)
    expert_output = experts_demo[expert_id](expert_input)   # (num_selected, d_model)

    # (c) 선택된 토큰들이 "이 전문가"에게 부여한 가중치를 찾습니다.
    #     matches[token_mask]        : 선택된 토큰들에 대해서만 (num_selected, top_k) True/False
    #     top_k_weights_demo[token_mask] : 그 토큰들의 top_k 가중치
    #     torch.topk는 한 토큰 안에서 같은 전문가를 두 번 뽑지 않으므로,
    #     토큰 하나당 matches에는 True가 최대 1개뿐입니다.
    #     따라서 곱해서 더하면 "이 전문가였던 자리"의 가중치 하나만 골라내는 효과를 냅니다.
    combine_weight = (
        top_k_weights_demo[token_mask] * matches[token_mask].float()
    ).sum(dim=-1, keepdim=True)   # (num_selected, 1)

    # (d) 가중치를 곱해서 원래 토큰 자리에 더해줍니다.
    output_demo[token_mask] += combine_weight * expert_output

print("\noutput_demo.shape:", output_demo.shape)  # (6, 8) - 입력과 동일한 shape
print(output_demo)


## 6. Shared Expert — 모든 토큰이 "항상" 거치는 공용 전문가

지금까지 만든 `output_demo`는 "라우팅된(선택된) 전문가들"만 반영한 값입니다.
DeepSeek-MoE 등 최근 구조들은 여기에 **라우팅과 상관없이 모든 토큰이 항상
통과하는 전문가**를 하나 더 두고, 그 출력을 더해줍니다.

- 라우팅되는 전문가들: 토큰마다 다른 전문가가 선택됨 (= "특화된" 지식 담당)
- Shared Expert: 모든 토큰에 항상 적용됨 (= "공통적으로" 필요한 지식 담당)

구조는 다른 전문가와 동일한 FFN이고, 다만 **게이팅(선택) 없이 항상 켜져
있다**는 점만 다릅니다.


In [ ]:
shared_expert_demo = nn.Sequential(
    nn.Linear(d_model_demo, d_ff_demo),
    nn.SiLU(),
    nn.Linear(d_ff_demo, d_model_demo),
)

# 라우팅 여부와 상관없이 "모든" 토큰(x_demo 전체)에 적용합니다.
shared_output_demo = shared_expert_demo(x_demo)

final_output_demo = output_demo + shared_output_demo

print("shared_output_demo.shape:", shared_output_demo.shape)  # (6, 8)
print("final_output_demo.shape :", final_output_demo.shape)   # (6, 8)


## 7. 지금까지의 조각들을 하나의 `nn.Module`로 합치기

위에서 한 것을 순서대로 정리하면:

1. `router`로 전문가별 점수 계산
2. `torch.topk`로 top_k개 전문가 선택
3. 선택된 점수만 `softmax` → 가중치
4. 전문가별로 담당 토큰만 모아 계산 후, 가중치를 곱해 원래 자리에 합산
5. `shared_expert` 출력을 모든 토큰에 더함

이 다섯 단계를 `__init__`(레이어들을 준비)과 `forward`(위 순서대로 실행) 두
메서드를 가진 `MoELayer` 클래스 하나로 정리하면 아래와 같습니다. 지금까지 본
코드와 한 줄 한 줄 비교하면서 읽어보세요 — 새로운 내용은 없고, 방금 한 것을
재사용 가능한 모듈로 "포장"한 것뿐입니다.

**참고**: 전문가들을 담을 때 파이썬 기본 리스트(`[]`)가 아니라
`nn.ModuleList`를 사용합니다. 일반 리스트에 담으면 PyTorch가 그 안의
파라미터를 인식하지 못해서, 학습(`.parameters()`, `.to(device)` 등)이
제대로 되지 않습니다.


In [ ]:
class MoELayer(nn.Module):
    """
    Mixture of Experts (MoE) 레이어.

    구성 요소
    ---------
    router        : 토큰마다 전문가별 점수(logit)를 매기는 nn.Linear
    experts       : num_experts개의 독립적인 FFN (Linear -> SiLU -> Linear)
    shared_expert : 라우팅과 무관하게 모든 토큰에 항상 적용되는 FFN (DeepSeek-MoE 스타일)

    입력/출력
    ---------
    입력  x : (batch, seq, d_model)
    출력    : (batch, seq, d_model)   <- 입력과 동일한 shape (레이어를 여러 번 쌓을 수 있어야 하므로)
    """

    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.d_model = d_model
        self.num_experts = num_experts
        self.top_k = top_k

        # 1) Router: d_model -> num_experts. 전문가 수만큼의 점수를 출력합니다.
        self.router = nn.Linear(d_model, num_experts)

        # 2) Experts: 서로 다른 가중치를 갖는 작은 FFN을 num_experts개 준비합니다.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.SiLU(),
                nn.Linear(d_ff, d_model),
            )
            for _ in range(num_experts)
        ])

        # 3) Shared Expert: 구조는 다른 전문가와 동일하지만, 라우팅 없이 항상 사용됩니다.
        self.shared_expert = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.SiLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        # x: (batch, seq, d_model)
        batch, seq, d = x.shape

        # 라우팅은 토큰 단위로 이루어지므로, (batch, seq)를 하나의 축으로 펼칩니다.
        # view 대신 reshape을 쓰면 x가 메모리상 연속(contiguous)이 아니어도 안전합니다.
        x_flat = x.reshape(-1, d)  # (batch*seq, d_model) = (num_tokens, d_model)

        # ---- 1) Router: 토큰별 전문가 점수 ----
        router_logits = self.router(x_flat)  # (num_tokens, num_experts)

        # ---- 2) Top-k 선택 ----
        top_k_logits, top_k_idx = torch.topk(router_logits, self.top_k, dim=-1)
        # top_k_logits, top_k_idx: 둘 다 (num_tokens, top_k)

        # ---- 3) 선택된 logit만 softmax -> 합이 1인 가중치 ----
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # (num_tokens, top_k)

        # ---- 4) 전문가별로 담당 토큰만 모아서 계산 (sparse 연산) ----
        output = torch.zeros_like(x_flat)  # (num_tokens, d_model)

        for expert_id in range(self.num_experts):
            # 이 전문가가 각 토큰의 top-k 안에 들어갔는지 (num_tokens, top_k)
            matches = (top_k_idx == expert_id)
            # 토큰마다 "이 전문가가 뽑혔는가"만 남긴 (num_tokens,) 마스크
            token_mask = matches.any(dim=-1)

            if not token_mask.any():
                # 이 전문가를 아무도 선택하지 않았다면 계산을 건너뜁니다 (연산 절약).
                continue

            # 이 전문가를 선택한 토큰들만 모아서 forward (나머지 토큰은 계산에서 제외!)
            expert_input = x_flat[token_mask]                       # (n_i, d_model)
            expert_output = self.experts[expert_id](expert_input)   # (n_i, d_model)

            # torch.topk는 같은 행에서 같은 인덱스를 두 번 뽑지 않으므로,
            # 토큰 하나당 matches에서 True는 최대 1개입니다.
            # 따라서 아래 sum은 "그 토큰이 이 전문가에게 준 가중치" 하나만 골라내는 효과를 냅니다.
            combine_weight = (
                top_k_weights[token_mask] * matches[token_mask].float()
            ).sum(dim=-1, keepdim=True)                              # (n_i, 1)

            # 가중치 x 전문가 출력을, 원래 토큰이 있던 자리에 더해줍니다.
            output[token_mask] += combine_weight * expert_output

        # ---- 5) Shared Expert: 모든 토큰에 항상 더해줌 ----
        output = output + self.shared_expert(x_flat)

        # 처음 shape (batch, seq, d_model)으로 되돌립니다.
        return output.reshape(batch, seq, d)


## 테스트 1 — 작은 예시(toy example)로 동작 확인

이 노트북 초반에 썼던 작은 하이퍼파라미터로 `MoELayer`를 만들고, `batch`와
`seq` 축이 있는 진짜 입력(3차원 텐서)을 넣어 shape이 잘 유지되는지 확인합니다.


In [ ]:
# batch=2, seq=3 인 입력을 만들어봅니다 (총 토큰 수 = 2*3 = 6, 위 예시와 동일)
batch_demo, seq_demo = 2, 3
x_demo_3d = torch.randn(batch_demo, seq_demo, d_model_demo)

moe_demo = MoELayer(
    d_model=d_model_demo,
    d_ff=d_ff_demo,
    num_experts=num_experts_demo,
    top_k=top_k_demo,
)

out_demo_3d = moe_demo(x_demo_3d)

print("입력 shape:", x_demo_3d.shape)   # (2, 3, 8)
print("출력 shape:", out_demo_3d.shape)  # (2, 3, 8) - 입력과 동일해야 함!
assert out_demo_3d.shape == x_demo_3d.shape, "MoE 레이어는 입력과 출력의 shape이 같아야 합니다!"
print("입력과 출력의 shape이 동일합니다.")


## 테스트 2 — 조금 더 실제와 가까운 크기로 확인

이제 논문/실전에서 자주 보이는 크기에 가까운 하이퍼파라미터로도 확인해봅니다.
(원본 실습 코드에서 사용한 값과 동일합니다.)

- `d_model = 512`
- `d_ff = 2048`
- `num_experts = 8`
- `top_k = 2`
- 입력: `batch=2, seq=10`


In [ ]:
moe = MoELayer(d_model=512, d_ff=2048, num_experts=8, top_k=2)

x = torch.randn(2, 10, 512)  # (batch=2, seq=10, d_model=512)
out = moe(x)

print(f"입력 shape    : {x.shape}")
print(f"MoE 출력 shape: {out.shape}")
assert out.shape == x.shape
print("입력과 출력의 shape이 동일합니다.")


## 보너스 1 — 전문가마다 토큰을 몇 개씩 받았을까? (부하 분산 살짝 엿보기)

라우터가 아직 학습되지 않은(방금 랜덤 초기화된) 상태이기 때문에, 지금은
사실상 "무작위로" 전문가를 고르는 것과 비슷합니다. 그래도 위에서 만든 `moe`의
라우팅 결과를 다시 꺼내서, 전문가 8명이 각각 몇 개의 토큰을 받았는지
세어볼 수 있습니다.

실전에서는 특정 전문가에만 토큰이 몰리는 "쏠림 현상"이 자주 발생하는데, 이를
막기 위한 장치(load balancing loss, capacity factor 등)가 따로 필요합니다.
이 노트북에서 직접 구현하지는 않지만, 왜 필요한지 감을 잡기 위해 분포만
확인해봅니다.


In [ ]:
# forward()와 완전히 동일한 라우팅 계산을 "구경"하기 위해 밖에서 한 번 더 해봅니다.
# (moe.router는 forward() 안에서 쓰인 것과 완전히 같은 가중치를 가진 동일한 모듈이므로,
#  같은 입력 x에 대해서는 forward() 내부에서 계산된 것과 정확히 같은 라우팅 결과가 나옵니다)
x_flat_check = x.reshape(-1, moe.d_model)
logits_check = moe.router(x_flat_check)
_, idx_check = torch.topk(logits_check, moe.top_k, dim=-1)  # (num_tokens, top_k)

# bincount(t, minlength=n): t 안에 0 ~ n-1 사이의 정수가 각각 몇 번 등장하는지 세어줍니다.
token_counts = torch.bincount(idx_check.flatten(), minlength=moe.num_experts)

total_tokens = x_flat_check.shape[0]
for expert_id, count in enumerate(token_counts.tolist()):
    print(f"전문가 {expert_id}: {count}개 토큰 담당 (전체 토큰 {total_tokens}개 중)")


In [ ]:
import matplotlib.pyplot as plt

# 그래프 안 글자가 한글 폰트 미설치 환경(Colab 기본 등)에서 깨질 수 있어 라벨은 영어로 표기합니다.
plt.figure(figsize=(6, 4))
plt.bar(range(moe.num_experts), token_counts.tolist())
plt.xlabel("Expert ID")
plt.ylabel("Number of tokens")
plt.title("Tokens routed to each expert (before training)")
plt.xticks(range(moe.num_experts))
plt.show()


## 보너스 2 — "전체 파라미터 수" vs "토큰 1개당 실제로 쓰이는 파라미터 수"

MoE를 쓰는 가장 큰 이유가 바로 이 숫자 차이입니다. 전문가를 늘릴수록 모델이
가진 전체 파라미터(=지식을 담을 수 있는 용량)는 늘어나지만, 토큰 하나를
처리할 때 실제로 연산에 쓰이는(활성화되는) 파라미터는 `top_k`에 의해서만
결정되므로 크게 늘지 않습니다.


In [ ]:
def count_params(module: nn.Module) -> int:
    # numel() = 텐서에 들어있는 원소(파라미터) 개수
    return sum(p.numel() for p in module.parameters())

router_params = count_params(moe.router)
one_expert_params = count_params(moe.experts[0])  # 전문가들은 구조가 같으므로 1명 것만 세도 충분
shared_params = count_params(moe.shared_expert)

total_expert_params = one_expert_params * moe.num_experts
total_params = router_params + total_expert_params + shared_params

# 토큰 1개를 처리할 때 실제로 지나가는 부분: router 전체 + 전문가 top_k명 + shared expert
active_params_per_token = router_params + one_expert_params * moe.top_k + shared_params

print(f"전문가 1명 파라미터 수          : {one_expert_params:,}")
print(f"전체 전문가({moe.num_experts}명) 파라미터 수 : {total_expert_params:,}")
print(f"라우터 파라미터 수              : {router_params:,}")
print(f"공유 전문가 파라미터 수         : {shared_params:,}")
print(f"모델 전체 파라미터 수           : {total_params:,}")
print(f"토큰 1개당 실제 사용 파라미터 수 : {active_params_per_token:,}")
print(f"활성화 비율 (active / total)     : {active_params_per_token / total_params:.1%}")


## 정리

- **Router**: 토큰마다 전문가별 점수를 매기는 `nn.Linear` 한 층
- **Top-k**: 점수가 높은 k개 전문가만 선택 (나머지는 계산 자체를 하지 않음 → sparse)
- **softmax(top-k)**: 선택된 것들끼리만 합이 1이 되는 가중치 생성
- **전문가별 계산**: 각 전문가는 "자기를 선택한 토큰"만 모아서 처리한 뒤, 가중치를
  곱해 원래 자리에 더함 (`output[token_mask] += weight * expert_output`)
- **Shared Expert**: 라우팅과 무관하게 모든 토큰에 항상 더해지는 공용 FFN
- 전문가 수를 늘려도 `top_k`가 고정이면, 토큰 1개당 "실제로 활성화되는"
  파라미터 수는 크게 늘지 않습니다 — 이것이 MoE가 "파라미터는 많지만 계산은
  가볍다"고 불리는 이유입니다.

**이 노트북에서 다루지 않은 것 (다음 실습에서 이어집니다)**
- Load balancing loss: 특정 전문가에 토큰이 쏠리지 않도록 유도하는 보조 손실 함수
- Capacity factor: 전문가 1명이 한 번에 처리할 수 있는 토큰 수 제한, 넘치면
  버리는(drop) 방식
- 여러 GPU에 전문가를 나눠 올리는 Expert Parallelism


## 연습 문제 (직접 실험해보기)

아래를 하나씩 바꿔보면서 shape/출력이 어떻게 달라지는지 확인해보세요.

1. `top_k`를 1로 바꾸면 어떻게 될까요? `top_k`를 `num_experts`와 같게
   (예: 4로) 바꾸면, 즉 전문가를 전부 다 쓰면 이 레이어는 무엇과 비슷해질까요?
2. `shared_expert`를 더하는 줄(`output = output + self.shared_expert(x_flat)`)을
   잠시 지우고 실행해보세요. 출력 shape은 여전히 유지되나요?
3. `num_experts=16, top_k=2`로 바꿔서 보너스 1의 "전문가별 토큰 수"를 다시
   세어보세요. 전문가 수가 늘어나면 전문가 1명이 받는 평균 토큰 수는 어떻게
   변할까요?
4. (심화) `token_mask`로 토큰을 걸러내지 않고, 그냥 모든 토큰을 모든 전문가에
   통과시킨 뒤 (선택되지 않은 전문가는 가중치 0을 곱해서) 더하면 결과가
   "수학적으로는" 같을까요? (힌트: 같습니다! 다만 전문가가 담당하지 않는
   토큰까지 계산하므로 훨씬 비효율적입니다.) 실제로 이렇게 구현해보고, 결과가
   `MoELayer`와 거의 같은 값이 나오는지 `torch.allclose`로 비교해보세요.
